# PushT Dataset Exploration — V0.1

This notebook analyzes the state/action/reward trajectory data used by the first Behavior Cloning baseline.

**Important:** V0.1 intentionally ignores RGB video and uses only the 2-D robot position. The block pose is hidden from this baseline.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from offline_rl_pusht.data.dataset import load_pusht_table, dataset_to_numpy


In [ ]:
ds = load_pusht_table()
arr = dataset_to_numpy(ds)
len(ds), ds.column_names


In [ ]:
df = pd.DataFrame({
    'episode': arr['episode_index'],
    'frame': arr['frame_index'],
    'state_x': arr['state'][:, 0],
    'state_y': arr['state'][:, 1],
    'action_x': arr['action'][:, 0],
    'action_y': arr['action'][:, 1],
    'reward': arr['reward'],
    'success': arr['success'],
})
df.head()


## Episode length distribution

In [ ]:
episode_lengths = df.groupby('episode').size()
print(episode_lengths.describe())
plt.figure(figsize=(7, 4))
plt.hist(episode_lengths.values, bins=25)
plt.xlabel('Episode length')
plt.ylabel('Count')
plt.title('PushT episode-length distribution')
plt.show()


## Reward distribution

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(df['reward'], bins=40)
plt.xlabel('Reward')
plt.ylabel('Frames')
plt.title('Per-frame reward distribution')
plt.show()


## State and action distributions

In [ ]:
for col in ['state_x', 'state_y', 'action_x', 'action_y']:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col], bins=40)
    plt.xlabel(col)
    plt.ylabel('Frames')
    plt.title(f'{col} distribution')
    plt.show()


## Example end-effector trajectories

In [ ]:
episode_ids = sorted(df['episode'].unique())[:8]
plt.figure(figsize=(7, 7))
for ep in episode_ids:
    sub = df[df['episode'] == ep]
    plt.plot(sub['state_x'], sub['state_y'], label=f'ep {ep}')
plt.xlabel('agent x')
plt.ylabel('agent y')
plt.title('Example agent trajectories')
plt.legend()
plt.axis('equal')
plt.show()


## Episode-level outcome summary

In [ ]:
episode_summary = df.groupby('episode').agg(
    length=('frame', 'count'),
    max_reward=('reward', 'max'),
    return_sum=('reward', 'sum'),
    success=('success', 'max'),
)
episode_summary.describe(include='all')
